# Evaluation Experiments
PSNR/SSIM image quality metrics and classification metrics aggregation across all scenarios.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from evaluation.image_metrics import ImageMetricsEvaluator, compute_psnr, compute_ssim
from evaluation.classification_metrics import ClassificationMetricsEvaluator
from evaluation.evaluator import ExperimentEvaluator

logging.basicConfig(level=logging.INFO)
PROJECT_ROOT = Path('..').resolve()
OUTPUT_DIR = PROJECT_ROOT / 'output' / '06_evaluation'

In [ ]:
# Image quality evaluation
evaluator = ExperimentEvaluator(OUTPUT_DIR)
df_quality = evaluator.evaluate_image_quality()

print(f'Total comparisons: {len(df_quality)}')
print(df_quality.groupby('stage')[['psnr', 'ssim']].mean().round(4))

In [ ]:
# PSNR boxplot per stage
if not df_quality.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.boxplot(data=df_quality, x='stage', y='psnr', ax=axes[0])
    axes[0].set_title('PSNR by Processing Stage')
    axes[0].tick_params(axis='x', rotation=45)
    
    sns.boxplot(data=df_quality, x='stage', y='ssim', ax=axes[1])
    axes[1].set_title('SSIM by Processing Stage')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'psnr' / 'psnr_ssim_boxplot.png', dpi=150)
    plt.show()

In [ ]:
# Load classification results from all scenarios
scenario_csv = PROJECT_ROOT / 'output' / '05_classification' / 'scenario_summary.csv'
if scenario_csv.exists():
    df_cls = pd.read_csv(scenario_csv)
    print(df_cls.to_string(index=False))
else:
    print('Run classification notebook first.')
    df_cls = pd.DataFrame()

In [ ]:
# Classification metrics bar chart
if not df_cls.empty:
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    fig, axes = plt.subplots(1, len(metrics), figsize=(18, 5))
    
    for ax, metric in zip(axes, metrics):
        bars = ax.bar(df_cls['scenario'], df_cls[metric], color='steelblue', edgecolor='black')
        ax.set_title(metric.capitalize())
        ax.set_ylim(0, 1)
        ax.tick_params(axis='x', rotation=45)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
    
    plt.suptitle('Classification Metrics Across Scenarios', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'classification_metrics' / 'metrics_comparison.png', dpi=150)
    plt.show()

In [ ]:
# Export all results
evaluator.export_all()

# Generate text summary
report = evaluator.generate_summary_report()
print(report)

In [ ]:
# Single image PSNR/SSIM demo
import cv2
import numpy as np

orig_dir = PROJECT_ROOT / 'output' / '01_preprocessing' / 'normalized'
lpf_dir = PROJECT_ROOT / 'output' / '03_frequency_filtering' / 'butterworth_lpf'

orig_files = sorted(orig_dir.glob('*.png'))[:5]
print('Per-image PSNR/SSIM (original vs LPF):')
for f in orig_files:
    proc = lpf_dir / f.name
    if not proc.exists():
        continue
    o = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
    p = cv2.imread(str(proc), cv2.IMREAD_GRAYSCALE)
    psnr = compute_psnr(o, p)
    ssim = compute_ssim(o, p)
    print(f'  {f.name}: PSNR={psnr:.2f} dB, SSIM={ssim:.4f}')